In [11]:
# we need to run on a py file instead of a jupyter notebook otherwise multiprocessing will not work properly
import os
import sys

# agent.py / monte_carlo_tree_search.py / minimax.py import `Team2.data_processing`,
# so the repo root (the parent of Team2/) has to be on sys.path. This notebook lives in
# Team2/, so sys.path already covers `agent` and `model_files` themselves.
TEAM2_DIR = os.path.abspath("")
REPO_ROOT = os.path.dirname(TEAM2_DIR)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from agent import Agent, pit
from model_files.SLPolicyValueGPU import SLPolicyValueNetwork
import torch
import chess


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("using device:", device)

# these weights live under backend/, not in Team2/model_weights/
WEIGHTS_PATH = os.path.join(
    REPO_ROOT, "Team2", "model_weights", "lab_trained_epoch_1.pth"
)

model1 = SLPolicyValueNetwork().to(device)
checkpoint = torch.load(WEIGHTS_PATH, map_location=device)
model1.load_state_dict(checkpoint["model"])
model1.eval()

agent = Agent(policy_value_network=model1, c_puct=0.0, dirichlet_alpha=0.3, dirichlet_epsilon=0.0)
# epsilon is set to 0 for no noise

using device: cpu


In [16]:
#human vs model

board = chess.Board()
human_turn = 1
while not board.is_game_over():
    print(board, "\n")
    if human_turn == 1:
        move = input("enter a move in UCI format\n")
        if move == "q":
            sys.exit()
        try:
            board.push_uci(move)
            human_turn *= -1
        except:
            continue
    
    else:
        data = agent.select_move(game_state=board, num_simulations=100, temperature=0, debug=True)
        move, stats = data[0], data[1]
        board.push_uci(move)
        human_turn *= -1
print(board, "\n")
print(stats[0])

r n b q k b n r
p p p p p p p p
. . . . . . . .
. . . . . . . .
. . . . . . . .
. . . . . . . .
P P P P P P P P
R N B Q K B N R 

r n b q k b n r
p p p p p p p p
. . . . . . . .
. . . . . . . .
. . . . P . . .
. . . . . . . .
P P P P . P P P
R N B Q K B N R 

r n b q k b n r
p p p . p p p p
. . . p . . . .
. . . . . . . .
. . . . P . . .
. . . . . . . .
P P P P . P P P
R N B Q K B N R 

r n b q k b n r
p p p . p p p p
. . . p . . . .
. . . . . . . .
. . . P P . . .
. . . . . . . .
P P P . . P P P
R N B Q K B N R 

r n b q k b n r
p p p . p . p p
. . . p . . . .
. . . . . p . .
. . . P P . . .
. . . . . . . .
P P P . . P P P
R N B Q K B N R 



SystemExit: 

/Users/mohimohi/fix-my-elo/.venv/lib/python3.13/site-packages/IPython/core/interactiveshell.py:3756: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [4]:
board = chess.Board()
board.push_uci("e2e4")
board.push_uci("e7e5")
board.push_uci("f2f4")
stockfish_turn = 1
agent.stockfish.set_depth(15)
moves = []
move = None
while not board.is_game_over():
    print(board, "\n")
    if stockfish_turn == 1:
        agent.stockfish.set_fen_position(board.fen())
        print(agent.stockfish.get_evaluation()["value"]/100)
        move = agent.stockfish.get_best_move()
        print(move)
        board.push_uci(move)
        stockfish_turn *= -1
    
    else:
        move = agent.select_move(game_state=board, num_simulations=3200, temperature=0, debug=True)
        board.push_uci(move)
        stockfish_turn *= -1
    moves.append(move)
    print(moves)

r n b q k b n r
p p p p . p p p
. . . . . . . .
. . . . p . . .
. . . . P P . .
. . . . . . . .
P P P P . . P P
R N B Q K B N R 

0.63
e5f4
['e5f4']
r n b q k b n r
p p p p . p p p
. . . . . . . .
. . . . . . . .
. . . . P p . .
. . . . . . . .
P P P P . . P P
R N B Q K B N R 

move: g1f3, count: 2772.0
move: d2d4, count: 217.0
move: b1c3, count: 132.0
move: g1e2, count: 28.0
move: e4e5, count: 26.0
final eval:  -0.13633800857592168
['e5f4', 'g1f3']
r n b q k b n r
p p p p . p p p
. . . . . . . .
. . . . . . . .
. . . . P p . .
. . . . . N . .
P P P P . . P P
R N B Q K B . R 

0.68
g7g5
['e5f4', 'g1f3', 'g7g5']
r n b q k b n r
p p p p . p . p
. . . . . . . .
. . . . . . p .
. . . . P p . .
. . . . . N . .
P P P P . . P P
R N B Q K B . R 

move: d2d4, count: 2293.0
move: h2h4, count: 133.0
move: g2g4, count: 131.0
move: b1c3, count: 106.0
move: f3e5, count: 92.0
final eval:  0.1524719075908203
['e5f4', 'g1f3', 'g7g5', 'd2d4']
r n b q k b n r
p p p p . p . p
. . . . . . . .
. . . . . . p

KeyboardInterrupt: 

In [2]:
agent.agent_vs_stockfish(2, 3200, "pgn_files/demo_3200_sims_vs_depth16.pgn")

move: e7e5, count: 1013.0
move: c7c5, count: 544.0
move: e7e6, count: 472.0
move: c7c6, count: 298.0
move: b8c6, count: 189.0
final eval:  -0.2080028805723608
move: b8c6, count: 1666.0
move: g8f6, count: 254.0
move: d7d6, count: 215.0
move: d7d5, count: 128.0
move: c7c6, count: 92.0
final eval:  -0.07900753195298617
move: g8f6, count: 2339.0
move: a7a6, count: 370.0
move: f8c5, count: 51.0
move: d7d6, count: 49.0
move: g8e7, count: 48.0
final eval:  -0.039437999105706764
move: c6d4, count: 1346.0
move: a7a6, count: 511.0
move: f8c5, count: 271.0
move: f8b4, count: 226.0
move: f8d6, count: 221.0
final eval:  0.10546550688064474
move: d4b5, count: 797.0
move: d8e7, count: 691.0
move: f8c5, count: 666.0
move: c7c6, count: 632.0
move: f8e7, count: 229.0
final eval:  -0.027255437676905823
move: d7d6, count: 1222.0
move: a7a6, count: 712.0
move: f6e4, count: 447.0
move: c7c6, count: 342.0
move: f8e7, count: 186.0
final eval:  -0.017180679298925125
move: c7c6, count: 1455.0
move: a7a6, count: